# NCBI Disease NER: multi-seed sweep on Colab

Runtime → Change runtime type → **GPU**. Results are written straight to Google Drive, and `ner.sweep` skips runs whose `metrics.json` already exists, so after a disconnect just re-run all cells to resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS_DIR = '/content/drive/MyDrive/comp9444_ncbi_runs/baseline_hparams'
!mkdir -p {RUNS_DIR}

In [ ]:
BRANCH = 'unified-pipeline'  # switch to 'main' after the PR is merged
!rm -rf /content/group_project
!git clone -q -b {BRANCH} https://github.com/lofty14/bert-ner-baseline.git /content/group_project
%cd /content/group_project
!git log --oneline -1

In [ ]:
# keep Colab's preinstalled torch
!pip install -q --use-pep517 "transformers==5.22.0" "accelerate==1.15.0" "seqeval==1.2.2"
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.is_available() and torch.cuda.get_device_name(0))

In [ ]:
# quick smoke test (~1 min) before the long run
!python -m ner.train --model bert --seed 0 --train_limit 200 --eval_limit 200 --epochs 1 --output_dir /tmp/smoke

In [ ]:
# old-notebook hyper-parameters (lr 2e-5, batch 8, 3 epochs), dev-based epoch selection
!python -m ner.sweep --models bert biobert pubmedbert --seeds 1 2 3 --runs_dir {RUNS_DIR}

In [ ]:
!python -m ner.aggregate {RUNS_DIR} --out {RUNS_DIR}/results.md
!cat {RUNS_DIR}/results.md